# Questão 8 — Sistema de Recomendação

## Objetivo

Construir um motor de recomendação simples baseado em similaridade de comportamento de compra entre produtos.

O item de referência será:

**GPS Garmin Vortex Maré Drift**

## Abordagem

A recomendação será feita a partir de uma matriz de interação usuário × produto, onde:

- cada linha representa um cliente
- cada coluna representa um produto
- o valor será 1 quando o cliente tiver comprado o produto ao menos uma vez
- o valor será 0 caso contrário

Com essa matriz, será calculada a **similaridade de cosseno entre produtos**, permitindo identificar quais itens possuem padrão de compra mais semelhante entre os clientes.

## Fonte de dados

Serão utilizadas as tabelas analíticas da camada gold:

- `lh_nautical.04_gold.fct_vendas`
- `lh_nautical.04_gold.dim_produto`

In [0]:
# Bibliotecas utilizadas na construção do recomendador
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [0]:
# Leitura das tabelas
df_vendas = spark.table("lh_nautical.04_gold.fct_vendas")
df_produtos = spark.table("lh_nautical.04_gold.dim_produto")

# Conversão para pandas para facilitar a manipulação da matriz usuário-item
pdf_vendas = df_vendas.toPandas()
pdf_produtos = df_produtos.toPandas()

# Conferência inicial
print("Linhas fct_vendas:", len(pdf_vendas))
print("Linhas dim_produto:", len(pdf_produtos))

Linhas fct_vendas: 9895
Linhas dim_produto: 150


## Preparação da base

Como a fato de vendas contém os identificadores de cliente e produto, e a dimensão de produto contém o nome do item, será feita uma junção entre as duas tabelas para permitir a identificação do produto de referência dentro da matriz de similaridade.

In [0]:
# Junção entre fato vendas e dimensão de produto
# Necessária para obter o nome do produto de referência
df_base = pdf_vendas.merge(
    pdf_produtos[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

# Visualização inicial da base analítica
df_base.head()

,sale_id,customer_id,product_id,sale_date,quantity,receita_transacao_brl,custo_unitario_brl,custo_total_brl,prejuizo_brl,teve_prejuizo,cost_start_date,exchange_rate_date,exchange_rate,product_name
0,0,42,105,2023-09-10,11,3405.00,312.17,3433.83,28.83,True,2023-04-17,2023-09-08,4.983500,Cabo de Nylon Danforth Prime
1,1,3,136,2024-09-15,9,16873.90,2176.42,19587.76,2713.86,True,2023-10-20,2024-09-13,5.571700,Cabo de Nylon Bruce Flux Hydro
2,2,25,139,2024-08-13,7,9475.30,1354.81,9483.66,8.36,True,2020-10-30,2024-08-13,5.487500,Boia de Arqueamento Danforth Torque
3,4,20,23,2023-02-03,5,55893.00,10760.90,53804.50,0.00,False,2022-10-13,2023-02-03,5.103000,Piloto Automático Furuno Torque Peak
4,5,8,57,2024-02-12,4,451403.90,111852.26,447409.05,0.00,False,2022-12-16,2024-02-09,4.971700,Motor de Popa Honda Vector Kinetic 174HP


In [0]:
# Criação de indicador binário de compra
# Independentemente da quantidade comprada, basta saber se o cliente comprou o produto ao menos uma vez
df_base["comprou"] = 1

# Construção da matriz usuário × produto
# Linhas: customer_id
# Colunas: product_id
# Valores: 1 se comprou ao menos uma vez, 0 caso contrário
matriz_usuario_item = pd.pivot_table(
    df_base,
    index="customer_id",
    columns="product_id",
    values="comprou",
    aggfunc="max",
    fill_value=0
)

# Garantir tipo inteiro para facilitar leitura
matriz_usuario_item = matriz_usuario_item.astype(int)

matriz_usuario_item.head()

product_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,...,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150
customer_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,0,...,0,1,0,1,0,1,1,1,1,1,1,1,1,1,1,0,1,0,1,0,0,0,0,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,0,0
2,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,1,1,...,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1
3,1,1,1,1,1,1,1,0,1,1,1,1,1,0,1,0,0,1,0,0,1,0,1,1,1,1,0,1,1,1,1,1,1,1,0,0,1,1,0,1,...,1,1,1,1,0,1,0,1,1,1,0,1,1,1,0,0,1,1,1,1,1,1,1,1,0,1,0,1,0,1,0,1,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,1,0,0,1,1,1,1,1,1,0,0,0,1,1,0,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,...,1,1,1,0,1,1,0,1,0,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,0,1,0,1,1
5,1,1,0,1,1,1,1,1,1,1,1,1,0,1,0,0,0,1,1,1,0,1,0,0,1,1,0,0,1,0,1,1,1,0,1,0,1,0,1,1,...,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,1,0,0,1,0,1,1,1,1,1,1,1,0,1,1,1,1,0


## Cálculo da similaridade entre produtos

A matriz construída está no formato usuário × produto.  
Como a similaridade deve ser calculada entre produtos, será utilizada a matriz transposta, onde cada produto passa a ser representado por um vetor de clientes.

In [0]:
# Transposição da matriz para o formato produto × usuário
# Cada linha passa a representar um produto
matriz_produto_usuario = matriz_usuario_item.T

matriz_produto_usuario.head()

customer_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49
product_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,0,1,0,1,1,1,1,0,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1
2,0,1,1,1,1,0,1,1,1,1,0,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,0,0,0,0,0,1,1,1,1,1,0,1,1,1,0,0,1,1,1
3,1,1,1,0,0,1,1,1,0,0,0,1,1,1,1,0,0,0,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1,1,1,1,0,1,1,1,1,1
4,1,0,1,1,1,0,0,1,1,1,1,1,1,1,1,0,0,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1
5,1,1,1,1,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,1,1,1,0,1,1,0,1,1,0,0,1,1,1,1,1,0,0,0,1,0,1,0,1,1,1,1,1


In [0]:
# Cálculo da similaridade de cosseno entre os vetores dos produtos
similaridade = cosine_similarity(matriz_produto_usuario)

# Transformação do resultado em DataFrame para facilitar consulta
similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz_produto_usuario.index,
    columns=matriz_produto_usuario.index
)

similaridade_df.head()

product_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,...,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150
product_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.775058,0.737865,0.810191,0.748331,0.769484,0.775058,0.727825,0.711512,0.698771,0.892149,0.764217,0.759257,0.748331,0.846432,0.801784,0.764217,0.795133,0.801784,0.737865,0.790569,0.790569,0.737865,0.715626,0.814877,0.814877,0.850000,0.775000,0.795133,0.764217,0.769484,0.727825,0.775000,0.805807,0.829515,0.754673,0.843274,0.709952,0.790569,0.721605,...,0.843274,0.835510,0.786373,0.790569,0.825000,0.727825,0.681554,0.810191,0.825000,0.770675,0.715626,0.892149,0.770675,0.753819,0.737865,0.775058,0.805807,0.810191,0.711512,0.769484,0.805807,0.759257,0.748331,0.737865,0.743834,0.711512,0.727825,0.800000,0.825000,0.705024,0.795133,0.753819,0.775058,0.769484,0.721605,0.775058,0.872082,0.721605,0.727825,0.775058
2,0.775058,1.000000,0.704295,0.757865,0.714286,0.712931,0.771429,0.750290,0.788811,0.687256,0.799086,0.676123,0.666737,0.742857,0.712931,0.685714,0.704295,0.712931,0.714286,0.676123,0.676123,0.732467,0.732467,0.676763,0.844742,0.844742,0.748331,0.775058,0.767772,0.788811,0.685511,0.694713,0.721605,0.805867,0.808543,0.747018,0.816982,0.667894,0.676123,0.800000,...,0.732467,0.784931,0.724714,0.788811,0.801784,0.778078,0.637536,0.784931,0.775058,0.647339,0.647339,0.824863,0.706188,0.722501,0.704295,0.800000,0.778078,0.811998,0.760639,0.795192,0.750290,0.753702,0.771429,0.647952,0.685511,0.704295,0.750290,0.775058,0.775058,0.724714,0.795192,0.778078,0.771429,0.767772,0.685714,0.742857,0.712931,0.628571,0.750290,0.742857
3,0.737865,0.704295,1.000000,0.800641,0.704295,0.865181,0.732467,0.712396,0.777778,0.707107,0.813326,0.722222,0.743161,0.619780,0.811107,0.760639,0.694444,0.757033,0.704295,0.750000,0.777778,0.777778,0.750000,0.783349,0.806898,0.806898,0.764217,0.869626,0.757033,0.777778,0.784070,0.712396,0.737865,0.739795,0.797234,0.648181,0.750000,0.628619,0.722222,0.704295,...,0.722222,0.747265,0.743161,0.722222,0.737865,0.794595,0.658553,0.800641,0.764217,0.725324,0.725324,0.813326,0.696311,0.684996,0.750000,0.704295,0.712396,0.693889,0.750000,0.729996,0.712396,0.685994,0.676123,0.722222,0.811107,0.805556,0.739795,0.790569,0.764217,0.685994,0.675923,0.739795,0.732467,0.757033,0.704295,0.788811,0.729996,0.732467,0.739795,0.704295
4,0.810191,0.757865,0.800641,1.000000,0.757865,0.753310,0.757865,0.789747,0.773953,0.735980,0.830257,0.800641,0.741467,0.703732,0.779287,0.730798,0.720577,0.727334,0.703732,0.773953,0.800641,0.773953,0.773953,0.724743,0.800250,0.825258,0.784873,0.835510,0.753310,0.773953,0.805263,0.816072,0.784873,0.789747,0.815374,0.735980,0.827329,0.747757,0.720577,0.784931,...,0.747265,0.820513,0.741467,0.800641,0.759555,0.737097,0.632717,0.794872,0.784873,0.752618,0.724743,0.830257,0.752618,0.710772,0.747265,0.784931,0.763422,0.794872,0.747265,0.779287,0.737097,0.686544,0.730798,0.747265,0.831239,0.800641,0.763422,0.784873,0.810191,0.659082,0.831239,0.789747,0.730798,0.779287,0.730798,0.730798,0.805263,0.757865,0.763422,0.757865
5,0.748331,0.714286,0.704295,0.757865,1.000000,0.795192,0.742857,0.694713,0.760639,0.627495,0.773309,0.788811,0.695725,0.771429,0.740351,0.714286,0.732467,0.740351,0.685714,0.732467,0.760639,0.760639,0.704295,0.676763,0.765547,0.765547,0.694879,0.828510,0.767772,0.704295,0.740351,0.694713,0.801784,0.722501,0.730297,0.627495,0.647952,0.728612,0.704295,0.685714,...,0.704295,0.730798,0.608760,0.732467,0.721605,0.722501,0.637536,0.730798,0.775058,0.735612,0.735612,0.773309,0.735612,0.805867,0.704295,0.714286,0.722501,0.730798,0.760639,0.740351,0.778078,0.753702,0.771429,0.788811,0.740351,0.788811,0.750290,0.748331,0.748331,0.695725,0.795192,0.722501,0.714286,0.685511,0.685714,0.714286,0.767772,0.657143,0.722501,0

## Identificação do produto de referência

Antes de gerar o ranking, será identificado o `product_id` correspondente ao item **GPS Garmin Vortex Maré Drift**.

In [0]:
# Definição do produto de referência
produto_referencia = "GPS Garmin Vortex Maré Drift"

# Busca do product_id correspondente
produto_ref_row = pdf_produtos[
    pdf_produtos["product_name"] == produto_referencia
][["product_id", "product_name"]].drop_duplicates()

produto_ref_row

,product_id,product_name
26,27,GPS Garmin Vortex Maré Drift


In [0]:
# Extração do product_id do produto de referência
produto_ref_id = int(produto_ref_row["product_id"].iloc[0])

print("Produto de referência:", produto_referencia)
print("ID do produto de referência:", produto_ref_id)

Produto de referência: GPS Garmin Vortex Maré Drift
ID do produto de referência: 27


In [0]:
# Série com a similaridade do produto de referência em relação a todos os demais
ranking_similaridade = (
    similaridade_df[produto_ref_id]
    .reset_index()
)

# Ajuste de nomes das colunas
ranking_similaridade.columns = ["product_id", "similaridade"]

# Remoção do próprio produto de referência do ranking
ranking_similaridade = ranking_similaridade[
    ranking_similaridade["product_id"] != produto_ref_id
].copy()

# Ordenação decrescente da similaridade
ranking_similaridade = ranking_similaridade.sort_values(
    by="similaridade",
    ascending=False
)

In [0]:
# Inclusão do nome do produto para facilitar interpretação do ranking
ranking_similaridade = ranking_similaridade.merge(
    pdf_produtos[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

# Ranking final dos 5 produtos mais similares
top5_similares = ranking_similaridade.head(5)

top5_similares

,product_id,similaridade,product_name
0,94,0.869626,Motor de Popa Volvo Magnum 276HP
1,11,0.868037,GPS Furuno Swift Leviathan Poseidon
2,35,0.853913,Radar Furuno Swift
3,115,0.850000,Cabo de Nylon Delta Force Magnum Leviathan
4,1,0.850000,Transponder AIS Maré Magnum


## Ranking dos produtos mais similares

A tabela abaixo apresenta os 5 produtos com maior similaridade de compra em relação ao item de referência, desconsiderando o próprio GPS.

In [0]:
# Exibição final do ranking com foco analítico
top5_similares[["product_id", "product_name", "similaridade"]]

,product_id,product_name,similaridade
0,94,Motor de Popa Volvo Magnum 276HP,0.869626
1,11,GPS Furuno Swift Leviathan Poseidon,0.868037
2,35,Radar Furuno Swift,0.853913
3,115,Cabo de Nylon Delta Force Magnum Leviathan,0.850000
4,1,Transponder AIS Maré Magnum,0.850000


In [0]:
# Produto com maior similaridade ao item de referência
produto_mais_similar = top5_similares.iloc[0]

print("id_produto com maior similaridade:", int(produto_mais_similar["product_id"]))
print("produto recomendado:", produto_mais_similar["product_name"])
print("similaridade:", round(produto_mais_similar["similaridade"], 4))

id_produto com maior similaridade: 94
produto recomendado: Motor de Popa Volvo Magnum 276HP
similaridade: 0.8696


## Questão 8.2 — Validação

O `id_produto` com maior similaridade ao **GPS Garmin Vortex Maré Drift** foi **94**.

## Questão 8.3 — Explicação

### Como a matriz foi construída?

A matriz foi construída no formato usuário × produto, com `customer_id` nas linhas e `product_id` nas colunas. O valor de cada célula foi definido como 1 quando o cliente comprou aquele produto ao menos uma vez e 0 quando nunca comprou. A quantidade comprada foi desconsiderada, conforme exigido pela questão.


### O que significa a similaridade de cosseno nesse contexto?

A similaridade de cosseno mede o quão parecidos são dois produtos com base no padrão de clientes que os compraram. Se dois produtos forem comprados com frequência pelos mesmos clientes, seus vetores terão direção semelhante e a similaridade será mais alta. Nesse contexto, isso indica afinidade de comportamento de compra entre itens.

### Uma limitação desse método de recomendação

Uma limitação importante é que o método considera apenas coocorrência de compra e não incorpora contexto adicional, como ordem temporal, quantidade comprada, categoria do produto, preço ou perfil do cliente. Além disso, produtos com poucas compras podem gerar similaridades pouco estáveis.